# Inflation Data Summary: Reusable GARCH Workflow

This notebook is a portable rewrite of the earlier version. It avoids machine-specific paths and wraps the analysis into reusable functions.


## 1) Imports

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from arch import arch_model

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


## 2) Configuration

Update these values to reuse the notebook on different samples.


In [2]:
CONFIG = {
    "data_filename": "Aggregate_CPI_inflation_20230513.xls",
    "frequency": "quarterly",  # "quarterly" or "monthly"
    "start_year": 1970,            # set to None to use full sample
    "hac_lags": 3,
    "distribution": "normal",   # passed to arch_model
}

CONFIG

{'data_filename': 'Aggregate_CPI_inflation_20230513.xls',
 'frequency': 'quarterly',
 'start_year': 1970,
 'hac_lags': 3,
 'distribution': 'normal'}

## 3) Utility Functions

In [4]:
def find_data_file(filename: str, start: Path | None = None) -> Path:
    """Search current folder and parent folders for a file (including DataSummary/)."""
    start = (start or Path.cwd()).resolve()
    candidates = [start]
    candidates.extend(start.parents)

    for base in candidates:
        direct = base / filename
        nested = base / "DataSummary" / filename
        if direct.exists():
            return direct
        if nested.exists():
            return nested

    raise FileNotFoundError(
        f"Could not locate '{filename}' from {start} or its parent folders."
    )


def standardize_columns(df: pd.DataFrame, frequency: str) -> pd.DataFrame:
    """Map raw spreadsheet column names to canonical names."""
    canonical = {
        "Year": "Year",
        "Quarter": "Quarter",
        "Month": "Month",
    }

    for col in df.columns:
        c = str(col)
        low = c.lower()
        if "price index" in low:
            canonical[c] = "Price index"
        elif low.startswith("inflation") and "shock" not in low:
            canonical[c] = "Inflation"
        elif "expected inflation" in low or "forecast" in low:
            canonical[c] = "Forecasted inflation"
        elif "inflation shock" in low:
            canonical[c] = "Inflation shock"

    out = df.rename(columns=canonical).copy()

    required = ["Year", "Price index", "Inflation", "Forecasted inflation", "Inflation shock"]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")

    if frequency == "quarterly" and "Quarter" not in out.columns:
        raise ValueError("Quarterly data is missing the 'Quarter' column.")
    if frequency == "monthly" and "Month" not in out.columns:
        raise ValueError("Monthly data is missing the 'Month' column.")

    return out


def load_inflation_data(file_path: Path) -> dict[str, pd.DataFrame]:
    """Load quarterly/monthly sheets and create datetime index."""
    xl = pd.ExcelFile(file_path)
    data = {}

    for sheet in xl.sheet_names:
        raw = xl.parse(sheet, skiprows=2)
        sheet_low = sheet.lower()
        if "quarter" in sheet_low:
            freq = "quarterly"
        elif "month" in sheet_low:
            freq = "monthly"
        else:
            continue

        df = standardize_columns(raw, frequency=freq)

        if freq == "quarterly":
            periods = pd.PeriodIndex(
                year=df["Year"].astype(int),
                quarter=df["Quarter"].astype(int),
                freq="Q",
            )
            df.index = periods.to_timestamp(how="end")
        else:
            df.index = pd.to_datetime(
                {
                    "year": df["Year"].astype(int),
                    "month": df["Month"].astype(int),
                    "day": 1,
                }
            )

        data[freq] = df.sort_index()

    if not data:
        raise ValueError(f"No monthly/quarterly sheets found in {file_path}")

    return data


def prepare_sample(df: pd.DataFrame, start_year: int | None = None) -> pd.DataFrame:
    out = df.copy()
    if start_year is not None:
        out = out[out["Year"] >= start_year].copy()

    out["Inflation_lag_1"] = out["Inflation"].shift(1)
    out["Inflation_lag_2"] = out["Inflation"].shift(2)
    out["Forecasted_inflation_lag_1"] = out["Forecasted inflation"].shift(1)

    return out.dropna()


## 4) Model Specifications

In [5]:
MEAN_MODEL_SPECS = {
    "M0: Inflation shock series": None,
    "M1: Inflation ~ const + lag1 + forecast": ["Inflation_lag_1", "Forecasted inflation"],
    "M2: Inflation ~ const + lag1 + lag2 + forecast": ["Inflation_lag_1", "Inflation_lag_2", "Forecasted inflation"],
    "M3: Inflation ~ const + lag1 + lag2 + forecast + forecast_lag1": ["Inflation_lag_1", "Inflation_lag_2", "Forecasted inflation", "Forecasted_inflation_lag_1"],
}

VOL_MODEL_SPECS = [
    # GARCH(p,q)
    {"family": "GARCH", "name": "GARCH(1,1)", "vol": "GARCH", "p": 1, "o": 0, "q": 1},
    {"family": "GARCH", "name": "GARCH(2,1)", "vol": "GARCH", "p": 2, "o": 0, "q": 1},
    {"family": "GARCH", "name": "GARCH(1,2)", "vol": "GARCH", "p": 1, "o": 0, "q": 2},
    {"family": "GARCH", "name": "GARCH(2,2)", "vol": "GARCH", "p": 2, "o": 0, "q": 2},

    # EGARCH(p,o,q)
    {"family": "EGARCH", "name": "EGARCH(1,1,1)", "vol": "EGARCH", "p": 1, "o": 1, "q": 1},
    {"family": "EGARCH", "name": "EGARCH(2,1,1)", "vol": "EGARCH", "p": 2, "o": 1, "q": 1},
    {"family": "EGARCH", "name": "EGARCH(1,1,2)", "vol": "EGARCH", "p": 1, "o": 1, "q": 2},
    {"family": "EGARCH", "name": "EGARCH(1,2,1)", "vol": "EGARCH", "p": 1, "o": 2, "q": 1},
    {"family": "EGARCH", "name": "EGARCH(2,2,1)", "vol": "EGARCH", "p": 2, "o": 2, "q": 1},
    {"family": "EGARCH", "name": "EGARCH(2,1,2)", "vol": "EGARCH", "p": 2, "o": 1, "q": 2},
    {"family": "EGARCH", "name": "EGARCH(1,2,2)", "vol": "EGARCH", "p": 1, "o": 2, "q": 2},
    {"family": "EGARCH", "name": "EGARCH(2,2,2)", "vol": "EGARCH", "p": 2, "o": 2, "q": 2},

    # GJR-GARCH(p,q) represented as GARCH with asymmetric term o
    {"family": "GJR-GARCH", "name": "GJR-GARCH(1,1)", "vol": "GARCH", "p": 1, "o": 1, "q": 1},
    {"family": "GJR-GARCH", "name": "GJR-GARCH(1,2)", "vol": "GARCH", "p": 1, "o": 1, "q": 2},
    {"family": "GJR-GARCH", "name": "GJR-GARCH(2,1)", "vol": "GARCH", "p": 2, "o": 2, "q": 1},
    {"family": "GJR-GARCH", "name": "GJR-GARCH(2,2)", "vol": "GARCH", "p": 2, "o": 2, "q": 2},
]


## 5) Estimation Functions

In [6]:
def fit_mean_model(sample: pd.DataFrame, regressors: list[str] | None, hac_lags: int = 3):
    """Return residual series and OLS results (if applicable)."""
    if regressors is None:
        resid = sample["Inflation shock"].dropna()
        return resid, None

    X = sm.add_constant(sample[regressors])
    y = sample["Inflation"]
    ols = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    resid = pd.Series(ols.resid, index=sample.index, name="resid").dropna()
    return resid, ols


def fit_one_vol_model(residuals: pd.Series, spec: dict, dist: str = "normal"):
    """Fit one ARCH-family model and return fit object + summary row."""
    kwargs = {
        "mean": "Constant",
        "vol": spec["vol"],
        "p": spec["p"],
        "q": spec["q"],
        "dist": dist,
    }
    if spec["o"] > 0:
        kwargs["o"] = spec["o"]

    fitted = arch_model(residuals, **kwargs).fit(disp="off")
    row = {
        "family": spec["family"],
        "model": spec["name"],
        "p": spec["p"],
        "o": spec["o"],
        "q": spec["q"],
        "aic": fitted.aic,
        "bic": fitted.bic,
        "loglikelihood": fitted.loglikelihood,
        "nobs": int(fitted.nobs),
    }
    return fitted, row


def evaluate_vol_models(residuals: pd.Series, specs: list[dict], dist: str = "normal"):
    rows = []
    fits = {}

    for spec in specs:
        try:
            fitted, row = fit_one_vol_model(residuals, spec=spec, dist=dist)
            rows.append(row)
            fits[spec["name"]] = fitted
        except Exception as exc:
            rows.append({
                "family": spec["family"],
                "model": spec["name"],
                "p": spec["p"],
                "o": spec["o"],
                "q": spec["q"],
                "aic": np.nan,
                "bic": np.nan,
                "loglikelihood": np.nan,
                "nobs": len(residuals),
                "error": str(exc),
            })

    table = pd.DataFrame(rows).sort_values(["aic", "bic"], na_position="last").reset_index(drop=True)
    return table, fits


def run_full_workflow(sample: pd.DataFrame, mean_specs: dict, vol_specs: list[dict], hac_lags: int, dist: str):
    all_rows = []
    objects = {}

    for mean_label, regressors in mean_specs.items():
        resid, ols = fit_mean_model(sample, regressors=regressors, hac_lags=hac_lags)
        table, fits = evaluate_vol_models(resid, specs=vol_specs, dist=dist)

        table.insert(0, "mean_model", mean_label)
        all_rows.append(table)

        best_name = table.iloc[0]["model"]
        objects[mean_label] = {
            "regressors": regressors,
            "mean_result": ols,
            "residuals": resid,
            "vol_table": table,
            "vol_fits": fits,
            "best_vol_name": best_name,
            "best_vol_fit": fits.get(best_name),
        }

    comparison = pd.concat(all_rows, ignore_index=True)
    return comparison, objects


## 6) Run Analysis

In [7]:
data_path = find_data_file(CONFIG["data_filename"])
all_data = load_inflation_data(data_path)
base_data = all_data[CONFIG["frequency"]]
sample_data = prepare_sample(base_data, start_year=CONFIG["start_year"])

print(f"Data file: {data_path}")
print(f"Frequency: {CONFIG['frequency']}")
print(f"Sample range: {sample_data.index.min()} to {sample_data.index.max()}")
print(f"Number of rows used: {len(sample_data)}")

sample_data.head()

ImportError: Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install xlrd.

In [8]:
comparison_table, model_objects = run_full_workflow(
    sample=sample_data,
    mean_specs=MEAN_MODEL_SPECS,
    vol_specs=VOL_MODEL_SPECS,
    hac_lags=CONFIG["hac_lags"],
    dist=CONFIG["distribution"],
)

comparison_table.head(20)

NameError: name 'sample_data' is not defined

## 7) Best Model Per Mean Specification

In [ ]:
best_models = (
    comparison_table
    .dropna(subset=["aic"])
    .sort_values(["mean_model", "aic", "bic"])
    .groupby("mean_model", as_index=False)
    .first()[["mean_model", "family", "model", "aic", "bic", "loglikelihood", "nobs"]]
)

best_models

## 8) Optional: Inspect Full Summaries

In [ ]:
# Example: show the best volatility model summary for each mean model.
# Run this cell when you want full parameter estimates.

for mean_label, obj in model_objects.items():
    print("\n" + "=" * 80)
    print(mean_label)
    print("Best volatility model:", obj["best_vol_name"])

    if obj["mean_result"] is not None:
        print("\nMean model (OLS with HAC) summary:")
        print(obj["mean_result"].summary())

    if obj["best_vol_fit"] is not None:
        print("\nVolatility model summary:")
        print(obj["best_vol_fit"].summary())


## Notes

- This rewrite keeps your original model families (GARCH, EGARCH, GJR-GARCH) and lag structures.
- To change sample period, edit `CONFIG["start_year"]`.
- To analyze monthly data, set `CONFIG["frequency"] = "monthly"`.
